# Lab 1: Introduction to Python & Backpropagation using Python



## Part A — Implementation

### Dataset

The dataset contains three normalized sensor measurements:

1. Temperature deviation
2. Vibration level
3. Pressure deviation

The target is:

- `1` → Acceptable
- `0` → Defective

This is a small educational dataset designed to demonstrate the complete backpropagation process.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Input features:
# [temperature deviation, vibration level, pressure deviation]
X = np.array([
    [0.10, 0.10, 0.10],
    [0.20, 0.15, 0.25],
    [0.15, 0.30, 0.20],
    [0.30, 0.25, 0.35],
    [0.70, 0.75, 0.65],
    [0.80, 0.70, 0.85],
    [0.75, 0.90, 0.80],
    [0.90, 0.85, 0.95]
], dtype=float)

# Target: 1 = Acceptable, 0 = Defective
y = np.array([
    [1],
    [1],
    [1],
    [1],
    [0],
    [0],
    [0],
    [0]
], dtype=float)

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("\nInput dataset:")
print(X)
print("\nTarget:")
print(y.ravel())

Input shape: (8, 3)
Target shape: (8, 1)

Input dataset:
[[0.1  0.1  0.1 ]
 [0.2  0.15 0.25]
 [0.15 0.3  0.2 ]
 [0.3  0.25 0.35]
 [0.7  0.75 0.65]
 [0.8  0.7  0.85]
 [0.75 0.9  0.8 ]
 [0.9  0.85 0.95]]

Target:
[1. 1. 1. 1. 0. 0. 0. 0.]


## Network Architecture

The ANN used in the experiment is:

**3 input neurons → 4 hidden neurons → 1 output neuron**

- Hidden layer: Sigmoid activation
- Output layer: Sigmoid activation
- Loss: Binary Cross-Entropy
- Optimization: Gradient Descent
- Default learning rate: `0.1`
- Default epochs: `5000`

In [2]:
INPUT_SIZE = X.shape[1]
HIDDEN_SIZE = 4
OUTPUT_SIZE = 1

LEARNING_RATE = 0.1
EPOCHS = 5000

# Random initialization
np.random.seed(SEED)

W1 = np.random.randn(INPUT_SIZE, HIDDEN_SIZE) * 0.5
b1 = np.random.randn(1, HIDDEN_SIZE) * 0.5

W2 = np.random.randn(HIDDEN_SIZE, OUTPUT_SIZE) * 0.5
b2 = np.random.randn(1, OUTPUT_SIZE) * 0.5

print("Initial W1 (Input -> Hidden):")
print(W1)
print("\nInitial b1:")
print(b1)

print("\nInitial W2 (Hidden -> Output):")
print(W2)
print("\nInitial b2:")
print(b2)

Initial W1 (Input -> Hidden):
[[ 0.248357 -0.069132  0.323844  0.761515]
 [-0.117077 -0.117068  0.789606  0.383717]
 [-0.234737  0.27128  -0.231709 -0.232865]]

Initial b1:
[[ 0.120981 -0.95664  -0.862459 -0.281144]]

Initial W2 (Hidden -> Output):
[[-0.506416]
 [ 0.157124]
 [-0.454012]
 [-0.706152]]

Initial b2:
[[0.732824]]


## Activation and Loss Functions

### Sigmoid

The sigmoid function maps values to the range `(0, 1)`, making it suitable for binary classification.

### Binary Cross-Entropy

The binary cross-entropy loss measures the difference between the actual binary target and the predicted probability.

In [3]:
def sigmoid(z):
    # Clipping prevents numerical overflow in exp()
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_derivative_from_output(a):
    return a * (1.0 - a)

def binary_cross_entropy(y_true, y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(
        y_true * np.log(y_pred) +
        (1 - y_true) * np.log(1 - y_pred)
    )

## Forward Propagation

For each training example:

1. Compute the hidden-layer weighted sum.
2. Apply sigmoid to obtain hidden activations.
3. Compute the output-layer weighted sum.
4. Apply sigmoid to obtain the final prediction.

The output is interpreted as the probability that the product is **Acceptable (1)**.

In [4]:
def forward_propagation(X, W1, b1, W2, b2, hidden_activation="sigmoid"):
    Z1 = X @ W1 + b1

    if hidden_activation == "sigmoid":
        A1 = sigmoid(Z1)
    elif hidden_activation == "relu":
        A1 = np.maximum(0, Z1)
    else:
        raise ValueError("hidden_activation must be 'sigmoid' or 'relu'")

    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }

    return A2, cache

## Backpropagation

Backpropagation calculates how much each weight and bias contributed to the prediction error.

For the sigmoid output with binary cross-entropy:

`dZ2 = A2 - y`

Then the gradients are propagated backward through the network to obtain:

- `dW2`, `db2`
- `dW1`, `db1`

Finally, gradient descent updates the parameters:

`W = W - learning_rate × gradient`

In [5]:
def backward_propagation(X, y, W2, cache, hidden_activation="sigmoid"):
    Z1 = cache["Z1"]
    A1 = cache["A1"]
    A2 = cache["A2"]

    m = X.shape[0]

    # Output layer gradient for sigmoid + binary cross entropy
    dZ2 = A2 - y
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # Propagate error to hidden layer
    dA1 = dZ2 @ W2.T

    if hidden_activation == "sigmoid":
        dZ1 = dA1 * sigmoid_derivative_from_output(A1)
    elif hidden_activation == "relu":
        dZ1 = dA1 * (Z1 > 0).astype(float)

    dW1 = (X.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return dW1, db1, dW2, db2

## Complete Training Function

This function performs:

**Forward propagation → Loss calculation → Backpropagation → Weight update**

for the requested number of epochs.

In [6]:
def train_network(
    X, y,
    hidden_size=4,
    learning_rate=0.1,
    epochs=5000,
    hidden_activation="sigmoid",
    seed=42,
    print_progress=False
):
    np.random.seed(seed)

    input_size = X.shape[1]

    W1 = np.random.randn(input_size, hidden_size) * 0.5
    b1 = np.random.randn(1, hidden_size) * 0.5

    W2 = np.random.randn(hidden_size, 1) * 0.5
    b2 = np.random.randn(1, 1) * 0.5

    losses = []

    for epoch in range(epochs):
        # Forward propagation
        y_pred, cache = forward_propagation(
            X, W1, b1, W2, b2, hidden_activation
        )

        # Error / loss
        loss = binary_cross_entropy(y, y_pred)
        losses.append(loss)

        # Backpropagation
        dW1, db1, dW2, db2 = backward_propagation(
            X, y, W2, cache, hidden_activation
        )

        # Gradient descent
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        if print_progress and (
            epoch == 0 or
            (epoch + 1) % max(1, epochs // 10) == 0
        ):
            print(f"Epoch {epoch+1:5d}/{epochs} | Loss: {loss:.6f}")

    final_pred, final_cache = forward_propagation(
        X, W1, b1, W2, b2, hidden_activation
    )

    return {
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2,
        "losses": np.array(losses),
        "predictions": final_pred,
        "cache": final_cache
    }

# Part A — Train for 5000 Epochs

The following cell trains the required 4-neuron hidden-layer network for 5000 epochs.

In [7]:
model_5000 = train_network(
    X, y,
    hidden_size=4,
    learning_rate=0.1,
    epochs=5000,
    hidden_activation="sigmoid",
    seed=SEED,
    print_progress=True
)

print("\nInitial loss:", model_5000["losses"][0])
print("Final loss:", model_5000["losses"][-1])

Epoch     1/5000 | Loss: 0.658008
Epoch   500/5000 | Loss: 0.253299
Epoch  1000/5000 | Loss: 0.081823
Epoch  1500/5000 | Loss: 0.041542
Epoch  2000/5000 | Loss: 0.026390
Epoch  2500/5000 | Loss: 0.018864
Epoch  3000/5000 | Loss: 0.014478
Epoch  3500/5000 | Loss: 0.011648
Epoch  4000/5000 | Loss: 0.009688
Epoch  4500/5000 | Loss: 0.008260
Epoch  5000/5000 | Loss: 0.007177

Initial loss: 0.6580083552121636
Final loss: 0.007176923152156206


In [8]:
# Display final weights
print("Final W1 (Input -> Hidden):")
print(model_5000["W1"])

print("\nFinal b1:")
print(model_5000["b1"])

print("\nFinal W2 (Hidden -> Output):")
print(model_5000["W2"])

print("\nFinal b2:")
print(model_5000["b2"])

Final W1 (Input -> Hidden):
[[ 0.72746  -1.028193  2.604067  3.051095]
 [ 0.360948 -1.09139   3.11189   2.730707]
 [ 0.21487  -0.52489   1.731659  1.707737]]

Final b1:
[[-0.178657  0.906304 -3.655148 -3.673056]]

Final W2 (Hidden -> Output):
[[-0.047768]
 [ 2.898151]
 [-5.813922]
 [-5.93812 ]]

Final b2:
[[4.663605]]


In [9]:
# Predictions
probabilities = model_5000["predictions"]
predicted_classes = (probabilities >= 0.5).astype(int)

accuracy = np.mean(predicted_classes == y) * 100

prediction_table = pd.DataFrame({
    "Actual": y.ravel().astype(int),
    "Predicted Probability": probabilities.ravel(),
    "Predicted Class": predicted_classes.ravel()
})

display(prediction_table)
print(f"Training Accuracy: {accuracy:.2f}%")

NameError: name 'pd' is not defined

## Training Error Curve

In [ ]:
plt.figure(figsize=(9,5))
plt.plot(model_5000["losses"])
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Training Error vs Epochs — Sigmoid Hidden Layer")
plt.grid(alpha=0.3)
plt.show()

# Part B — Observation & Analysis

## Q1. What is the purpose of the forward propagation step?

Forward propagation passes the input data through the neural network from the input layer to the output layer. It calculates the hidden-layer activations and then the final prediction. The prediction is required before the error can be calculated and gradients can be computed.

## Q2. Why is the Sigmoid activation function used?

Sigmoid converts a neuron's weighted sum into a value between **0 and 1**. Since this experiment is a binary classification problem, the output can be interpreted as the probability that a product belongs to the **Acceptable** class.

Sigmoid is also differentiable, which allows its derivative to be used during backpropagation.

## Q3. Record the initial and final loss values. What do they indicate?

The notebook prints the exact values after training.

- **Initial loss:** `model_5000["losses"][0]`
- **Final loss:** `model_5000["losses"][-1]`

A substantial reduction in loss indicates that the network learned useful relationships between the sensor measurements and the target labels.

In [ ]:
print("Initial loss =", model_5000["losses"][0])
print("Final loss   =", model_5000["losses"][-1])
print("Loss reduction =", model_5000["losses"][0] - model_5000["losses"][-1])

## Q4. Why is Backpropagation considered the learning mechanism of an ANN?

Backpropagation computes the gradient of the loss with respect to each weight and bias by propagating the error backward through the network. Gradient descent then uses these gradients to update the parameters.

Repeated updates gradually reduce the prediction error, allowing the neural network to learn from the training examples.

## Q5. What happens if the learning rate is increased from 0.1 to 1.0?

A learning rate of `1.0` makes each parameter update much larger.

Possible observations are:

- Faster movement toward a solution at the beginning.
- The loss may oscillate instead of decreasing smoothly.
- The network can overshoot good parameter values.
- In extreme cases, training can become unstable.

The following experiment allows the observation to be made directly.

In [ ]:
model_lr_01 = train_network(
    X,y,hidden_size=4,learning_rate=0.1,epochs=5000,
    hidden_activation="sigmoid",seed=SEED
)

model_lr_10 = train_network(
    X,y,hidden_size=4,learning_rate=1.0,epochs=5000,
    hidden_activation="sigmoid",seed=SEED
)

print("Learning rate 0.1 final loss:", model_lr_01["losses"][-1])
print("Learning rate 1.0 final loss:", model_lr_10["losses"][-1])

plt.figure(figsize=(9,5))
plt.plot(model_lr_01["losses"], label="Learning rate = 0.1")
plt.plot(model_lr_10["losses"], label="Learning rate = 1.0")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Effect of Learning Rate")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Q6. Train using 100, 1000 and 5000 epochs

The table below is generated automatically from the actual runs.

**Prediction Accuracy** is calculated as:

`correct predictions / total samples × 100`

In [ ]:
def evaluate_epochs(epoch_list):
    rows = []

    for e in epoch_list:
        result = train_network(
            X, y,
            hidden_size=4,
            learning_rate=0.1,
            epochs=e,
            hidden_activation="sigmoid",
            seed=SEED
        )

        pred = (result["predictions"] >= 0.5).astype(int)
        acc = np.mean(pred == y) * 100

        rows.append({
            "Epochs": e,
            "Final Error": result["losses"][-1],
            "Prediction Accuracy (%)": acc
        })

    return pd.DataFrame(rows)

epoch_results = evaluate_epochs([100, 1000, 5000])
display(epoch_results.style.format({
    "Final Error":"{:.6f}",
    "Prediction Accuracy (%)":"{:.2f}"
}))

## Q7. What if the hidden layer contains 10 neurons instead of 4?

The network will have greater representational capacity, but it will **not always perform better**.

Ten neurons can learn more complex patterns, but the additional parameters can also:

- increase computational cost,
- make optimization more difficult,
- increase the risk of overfitting,
- provide little or no benefit when the dataset is already simple.

For a small, simple dataset, four neurons may already be sufficient.

## Q8. Why are weights initialized randomly instead of all being zero?

If all neurons in a layer start with exactly the same weights, they receive the same gradients and remain identical during training. This is called a **symmetry problem**.

Random initialization breaks this symmetry so different neurons can learn different features.

## Q9. Role of the learning rate

The learning rate controls the size of each gradient-descent update.

### Too small
- Training becomes very slow.
- Many epochs may be required.
- The model can appear to stop learning because progress is extremely small.

### Too large
- Updates can overshoot the minimum.
- Loss can oscillate.
- Training can become unstable or diverge.

## Q10. Three real-world applications of Backpropagation-based neural networks

1. **Manufacturing quality inspection** — classify products as defective or acceptable.
2. **Medical diagnosis** — classify patients or medical images based on learned patterns.
3. **Fraud detection** — identify suspicious financial transactions.

Other applications include speech recognition, image classification, predictive maintenance, recommendation systems, and handwriting recognition.

In [ ]:
# Train Sigmoid and ReLU hidden-layer versions.
sigmoid_result = train_network(
    X,y,hidden_size=4,learning_rate=0.1,epochs=5000,
    hidden_activation="sigmoid",seed=SEED
)

relu_result = train_network(
    X,y,hidden_size=4,learning_rate=0.1,epochs=5000,
    hidden_activation="relu",seed=SEED
)

def summarize(result, activation):
    pred = (result["predictions"] >= 0.5).astype(int)
    return {
        "Hidden Activation": activation,
        "Epochs": len(result["losses"]),
        "Final Loss": result["losses"][-1],
        "Accuracy (%)": np.mean(pred == y) * 100,
        "Predicted Classes": pred.ravel().tolist()
    }

activation_comparison = pd.DataFrame([
    summarize(sigmoid_result,"Sigmoid"),
    summarize(relu_result,"ReLU")
])

display(activation_comparison.style.format({
    "Final Loss":"{:.6f}",
    "Accuracy (%)":"{:.2f}"
}))

In [ ]:
# Compare convergence curves
plt.figure(figsize=(10,5))
plt.plot(sigmoid_result["losses"], label="Sigmoid Hidden Layer")
plt.plot(relu_result["losses"], label="ReLU Hidden Layer")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Sigmoid vs ReLU — Training Convergence")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Print predicted probabilities/classes
for name, result in [("Sigmoid",sigmoid_result),("ReLU",relu_result)]:
    probs = result["predictions"].ravel()
    classes = (probs >= 0.5).astype(int)
    print("\n", name)
    print("Probabilities:", np.round(probs,6))
    print("Classes      :", classes)